# AULA 27/10 — WEB SCRAPING E PRIMEIRO PIPELINE ETL
## Coleta responsável + funções reutilizáveis + carga em SQL

### Objetivos
Ao final da aula, você deverá conseguir:
- explicar o que é web scraping;
- identificar elementos HTML básicos;
- coletar dados de uma página usando Python;
- respeitar regras de acesso, `robots.txt` e limites do site;
- separar um processo em **Extract, Transform e Load**;
- criar funções reutilizáveis;
- gravar um DataFrame em um banco SQL simples.

> **Atenção:** nesta aula usamos primeiro um HTML local e depois, opcionalmente, um site criado especificamente para treinamento de scraping. Não use técnicas para contornar login, CAPTCHA, bloqueios ou restrições de acesso.

# 1. O que é web scraping?

Web scraping é a coleta automatizada de informações apresentadas em páginas web.

Uma sequência simplificada é:

**URL/HTML → requisição → HTML → seleção dos elementos → dados estruturados**

Antes de automatizar uma coleta, pergunte:
1. Existe API oficial?
2. O site permite esse acesso?
3. O `robots.txt` permite o caminho?
4. A frequência da coleta pode sobrecarregar o serviço?
5. Os dados contêm informações pessoais ou sensíveis?

# 2. Upload do HTML didático
Faça upload de `catalogo_produtos.html`.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving catalogo_produtos.html to catalogo_produtos.html


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

arquivo_html = next(iter(uploaded))

with open(arquivo_html, "r", encoding="utf-8") as arquivo:
    html = arquivo.read()

soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)

Catálogo Didático


## Como ler os seletores?

No HTML da aula existe:

```html
<article class="produto">
    <h2 class="nome">Mouse sem fio</h2>
    <span class="preco">R$ 89,90</span>
</article>
```

- `article` é uma **tag**.
- `produto` é uma **classe CSS**.
- `.produto` significa “elementos cuja classe é produto”.
- `.nome` seleciona o elemento com classe `nome` dentro de cada produto.

# 3. Encontrando os produtos

In [ ]:
# Transformando o texto HTML em uma estrutura de pesquisa
soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)

Catálogo Didático


In [ ]:
# Localizando produtos
produtos_html = soup.select(".produto")

print(
    "Quantidade encontrada: ",
    len(produtos_html))

Quantidade encontrada:  5


# 4. Extraindo um produto por partes

In [ ]:
# Extraindo um produto antes de automatizar a repetição

primeiro = produtos_html[0]
print(primeiro.prettify())

<article class="produto" data-id="P001">
 <h2 class="nome">
  Mouse sem fio
 </h2>
 <span class="categoria">
  Periféricos
 </span>
 <span class="preco">
  R$ 89,90
 </span>
 <span class="estoque">
  14
 </span>
 <a class="detalhe" href="/produto/P001">
  ver produto
 </a>
</article>



In [ ]:
# Extração dos campos

produto_id = primeiro.get("data-id")
nome = primeiro.select_one(".nome").get_text(strip=True)

categoria = primeiro.select_one(
    ".categoria"
).get_text(strip=True)

preco = primeiro.select_one(
    ".preco"
).get_text(strip=True)

estoque = primeiro.select_one(
    ".estoque"
).get_text(strip=True)

print(produto_id, nome, categoria, preco, estoque)

P001 Mouse sem fio Periféricos R$ 89,90 14


In [ ]:
# Tipos

print(type(preco))
print(type(estoque))

<class 'str'>
<class 'str'>


# 5. Transformando repetição em função

In [ ]:
# Aplicar a mesma regra a todos os produtos encontrados

def extrair_produtos(soup):
  registros = []

  for card in soup.select(".produto"):
    registros.append({
        "produto_id": card.get("data-id"),
        "nome": card.select_one(
            ".nome"
        ).get_text(strip=True),
        "categoria": card.select_one(
            ".categoria"
        ).get_text(strip=True),
        "preco_texto": card.select_one(
            ".preco"
        ).get_text(strip=True),
        "estoque_texto": card.select_one(
            ".estoque"
        ).get_text(strip=True)
    })

  return pd.DataFrame(registros)

In [ ]:
df_bruto = extrair_produtos(soup)

display(df_bruto)

,produto_id,nome,categoria,preco_texto,estoque_texto
0,P001,Mouse sem fio,Periféricos,"R$ 89,90",14
1,P002,Teclado mecânico,Periféricos,"R$ 249,90",7
2,P003,Webcam HD,Vídeo,"R$ 179,90",3
3,P004,Headset,Áudio,"R$ 199,90",9
4,P005,Hub USB-C,Acessórios,"R$ 159,90",5


In [ ]:
# Leitura dos dados

print(df_bruto.shape)
display(df_bruto.dtypes)

(5, 5)


,0
produto_id,object
nome,object
categoria,object
preco_texto,object
estoque_texto,object


# 6. TRANSFORM — limpando tipos

In [ ]:
# Transformar preço e estoque em colunas numéricas

def transformar_produtos(df):
  tratado = df.copy()

  tratado["preco"] = (
      tratado["preco_texto"]
      .str.replace("R$", "", regex=False)
      .str.replace(".", "", regex=False)
      .str.replace(",", ".", regex=False)
      .astype(float)
  )

  tratado["estoque"] = (
      tratado["estoque_texto"]
      .astype(int)
  )

  return tratado.drop(
      columns=["preco_texto", "estoque_texto"]
  )

In [ ]:
# Visualizando os dados numéricos tratados

df_tratado = transformar_produtos(df_bruto)
display(df_tratado)

,produto_id,nome,categoria,preco,estoque
0,P001,Mouse sem fio,Periféricos,89.9,14
1,P002,Teclado mecânico,Periféricos,249.9,7
2,P003,Webcam HD,Vídeo,179.9,3
3,P004,Headset,Áudio,199.9,9
4,P005,Hub USB-C,Acessórios,159.9,5


In [ ]:
# Tamanho da tabela transformada

print("Dimensões: ", df_tratado.shape)

Dimensões:  (5, 5)


In [ ]:
# Tipos de dados da tabela transformada

display(df_tratado.dtypes)

,0
produto_id,object
nome,object
categoria,object
preco,float64
estoque,int64


In [ ]:
# Valores ausentes da tabela transformada

display(df_tratado.isna().sum())

,0
produto_id,0
nome,0
categoria,0
preco,0
estoque,0


In [ ]:
# Valores duplicados na tabela transformada

print(
    "Identificadores repetidos: ",
    df_tratado["produto_id"].duplicated().sum()
)

Identificadores repetidos:  0


A gente uma tabela com as mesmas dimensões, 5 linhas e 5 colunas.

Preço com tipo numérico decimal;

Estoque com tipo numérico inteiro;

Nenhum valor ausente e nenhum valor duplicado.


# 7. LOAD — carregando em SQLite

## O que é ETL?

**Extrair - Transformar - Carregar**

ETL organiza o caminho que os dados percorrem. A extração traz os dados da origem. A transformação aplica as regras. A carga grava o resultado no destino.

## Web Scraping

É uma tecnica que usamos para extrair dados do HTML.

## Pipeline

É uma sequência organizada de etapas. A etapa produz o resultado que a próxima utiliza.

## SQL

Linguagem utilizada para consultar e manipular dados

## SQLite

Sistema de banco de dados que a gente vai utilizar

## catalogo.db

Arquivo que armazena o banco

## produtos

Nome da tabela dentro do banco

## Conexão

Objeto utilizado pelo Python para acessar esse banco

In [ ]:
def carregar_sql(df, caminho_banco="catalogo.db"):
  conexao = sqlite3.connect(caminho_banco)

  df.to_sql(
      "produtos",
      conexao,
      if_exists="replace",
      index=False
  )

  return conexao

con = carregar_sql(df_tratado)

## Consulta do banco usando SQL

In [ ]:
consulta = pd.read_sql_query(
    """
    SELECT categoria,
         COUNT(*) AS quantidade_produtos,
         ROUND(AVG(preco), 2) AS preco_medio

    FROM produtos
    GROUP BY categoria
    ORDER BY preco_medio DESC
    """,
    con
)

display(consulta)

,categoria,quantidade_produtos,preco_medio
0,Áudio,1,199.9
1,Vídeo,1,179.9
2,Periféricos,2,169.9
3,Acessórios,1,159.9


# Analise do resultado

A categoria Periféricos possui dois produtos. Seu preço médio é calculado usandou o mouse e o teclado: 89,90 + 249,90 / 2 = 169,90.



# 8. Agora você construiu um ETL

### Extract
`extrair_produtos()`

### Transform
`transformar_produtos()`

### Load
`carregar_sql()`

**fonte → extração → transformação → destino**

# 9. Introdução a classes

Uma classe permite reunir dados e comportamentos, e as suas instâncias podemn guarda um estado próprio.

**Função** = Organiza UMA tarefa

**Classe** = organiza UM CONJUNTO de tarefas e o estado do processo

In [ ]:
class PipelineCatalogo:
  def __init__(self, soup):
    self.soup = soup
    self.dados = None

  def extrair(self):
    self.dados = extrair_produtos(self.soup)
    return self

  def transformar(self):
    self.dados = transformar_produtos(self.dados)
    return self

In [ ]:
pipeline = PipelineCatalogo(soup)
pipeline.extrair()
display(pipeline.dados)

,produto_id,nome,categoria,preco_texto,estoque_texto
0,P001,Mouse sem fio,Periféricos,"R$ 89,90",14
1,P002,Teclado mecânico,Periféricos,"R$ 249,90",7
2,P003,Webcam HD,Vídeo,"R$ 179,90",3
3,P004,Headset,Áudio,"R$ 199,90",9
4,P005,Hub USB-C,Acessórios,"R$ 159,90",5


In [ ]:
pipeline.transformar()
display(pipeline.dados)

,produto_id,nome,categoria,preco,estoque
0,P001,Mouse sem fio,Periféricos,89.9,14
1,P002,Teclado mecânico,Periféricos,249.9,7
2,P003,Webcam HD,Vídeo,179.9,3
3,P004,Headset,Áudio,199.9,9
4,P005,Hub USB-C,Acessórios,159.9,5


In [ ]:
# Etapa de carga

class PipelineCatalogo:
  def __init__(self, soup):
    self.soup = soup
    self.dados = None

  def extrair(self):
    self.dados = extrair_produtos(self.soup)
    return self

  def transformar(self):
    self.dados = transformar_produtos(self.dados)
    return self

  def carregar(self, caminho_banco="catalogo_pipeline.db"):
     return carregar_sql(self.dados, caminho_banco=caminho_banco)
#

In [ ]:
pipeline = PipelineCatalogo(soup)
pipeline.extrair()
pipeline.transformar()

con_pipeline = pipeline.carregar()

display(pipeline.dados)

,produto_id,nome,categoria,preco,estoque
0,P001,Mouse sem fio,Periféricos,89.9,14
1,P002,Teclado mecânico,Periféricos,249.9,7
2,P003,Webcam HD,Vídeo,179.9,3
3,P004,Headset,Áudio,199.9,9
4,P005,Hub USB-C,Acessórios,159.9,5


# 10. Prática em site de treinamento

O site **Books to Scrape** foi criado como sandbox para aprendizagem.

In [27]:
import requests

url = "https://books.toscrape.com/"

resposta = requests.get(
    url,
    timeout=10,
    headers={
        "User-Agent": "AulaDados-T8/1.0"
    }
)

print("Status HTTP: ", resposta.status_code)
print("Quantidade de caracteres: ", len(resposta.text))

Status HTTP:  200
Quantidade de caracteres:  51294


In [28]:
if resposta.ok:
  soup_web = BeautifulSoup(
      resposta.text,
      "html.parser"
  )
  titulos = [
      item.get("title")
      for item in soup_web.select(
          "article.product_pod h3 a"
      )
  ]

  print(titulos[:5])

['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']


In [29]:
titulos = []

for item in soup_web.select("article.product_pod h3 a"):
  titulo = item.get("title")
  titulos.append(titulo)

  print(titulo[:5])

A Lig
Tippi
Soumi
Sharp
Sapie
The R
The D
The C
The B
The B
Starv
Shake
Set M
Scott
Rip i
Our B
Olio
Mesae
Liber
It's 


In [30]:
from bs4 import BeautifulSoup

soup_web = BeautifulSoup(resposta.text, "html.parser")

primeiro = soup_web.select_one("article.product_pod")
print(primeiro.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [31]:
print(soup_web.select("h3")[:3])

[<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>, <h3><a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a></h3>, <h3><a href="catalogue/soumission_998/index.html" title="Soumission">Soumission</a></h3>]


# DESAFIO FINAL

Crie um relatório com:
1. quantidade de produtos;
2. preço médio;
3. produto mais caro;
4. total de unidades em estoque;
5. categoria com maior preço médio.

Depois explique:
> **Qual parte foi Extract? Qual foi Transform? Qual foi Load?**

# Relatório do case de catálogo de produtos da página HTML.

## Quantidade de produtos

Quantos produtos existem?


In [33]:
# Registros do catalogo

quantidade_produtos = len(df_tratado)

print("Quantidade de produtos: ", quantidade_produtos)

Quantidade de produtos:  5


# Preço médio

In [34]:
# Reais por produto cadastrado

preco_medio = df_tratado["preco"].mean()

print(f"Preço médio: R$ {preco_medio: .2f}")

Preço médio: R$  175.90


# Produto mais caro

Nome e preço do produto

In [36]:
indice_mais_caro = df_tratado["preco"].idxmax()

produto_mais_caro = df_tratado.loc[
    indice_mais_caro,
    "nome"
]

preco_mais_caro = df_tratado.loc[
    indice_mais_caro,
    "preco"
]

print(
    f"Produto mais caro: {produto_mais_caro} "
    f"- R$ {preco_mais_caro: .2f}"
)

Produto mais caro: Teclado mecânico - R$  249.90


# Total de unidades em estoque

In [37]:
estoque_total = df_tratado["estoque"].sum()

print("Total de unidades em estoque: ", estoque_total)

Total de unidades em estoque:  38


# Categoria com maior preço médio

In [38]:
medias_por_categoria = (
    df_tratado.groupby("categoria")["preco"].mean()
)

categoria_maior_media = medias_por_categoria.idxmax()

maior_media = medias_por_categoria.loc[
    categoria_maior_media]

print(
     f"Categoria com maior preço médio: "
     f"{categoria_maior_media} "
     f"- R$ {maior_media: .2f}"
  )

Categoria com maior preço médio: Áudio - R$  199.90


# Relatório final da análise

O catálogo possui cinco produtos, com preço médio de R$ 175,90.

O teclado mecânico apresenta o maior preço individual, de R$ 249,90.

O estoque totaliza 38 unidades.

E a categoria Áudio apresenta a maior média de preço médio, de R$ 199,90.

Considerando o único produto dessa categoria presente no catálogo.

# Estoque de até 5 unidades

Quais estoques estão baixo e precisam de reposição?

In [39]:
estoque_baixo = df_tratado.loc[
    df_tratado["estoque"] <= 5,
    ["nome", "estoque"]
]

display(estoque_baixo)

,nome,estoque
2,Webcam HD,3
4,Hub USB-C,5


# Informação adicional do Relatório

Dois produtos merecem uma conferência de estoque. Uma decisão de reposição também precisaria considerar demanda e prazo de fornecimento.

In [41]:
df_tratado.to_csv(
    "catalogo_produtos_tratados.csv",
    index=False,
    encoding="utf-8"
)

In [42]:
con.close()

con_pipeline.close()